<a href="https://colab.research.google.com/github/tuan314159265/parking-occupancy/blob/main/notebooks/01_pklot_run_all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PKLot: GitHub → module Python → Colab

Phiên bản **drive-auth-v2**. Bản này nhận diện lỗi credential propagation, hướng dẫn cấp quyền và dừng trước khi tải dataset nếu Drive chưa kết nối.


In [ ]:
# Điền thông tin repo chứa parking_module.py trước khi Run All.
OWNER = "tuan314159265"
REPO = "parking-occupancy"
REF = "main"  # Đổi thành master nếu cần; dùng commit SHA để tái lập.
MODULE_PATH = "parking_module.py"

from pathlib import Path
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = "parking-occupancy"
DOWNLOAD_DATA = True  # False: archive phải có sẵn và hoàn chỉnh.
REUSE_EXTRACTED = True  # Dùng lại dữ liệu đã giải nén thành công.


## Kết nối Google Drive

Trong cửa sổ cấp quyền Google, chọn tài khoản Drive của bạn, đánh dấu **tất cả các quyền được yêu cầu** (hoặc **Select all**) rồi bấm **Continue**. Bỏ sót một quyền có thể gây lỗi `credential propagation was unsuccessful`. Không chỉ chọn riêng quyền đọc/ghi Drive.

Nếu không thấy cửa sổ, kiểm tra popup bị chặn. Nếu vẫn lỗi dù đã cấp đủ quyền, thử Runtime → Disconnect and delete runtime rồi kết nối lại. Thao tác này xóa dữ liệu tạm của runtime, giữ file trên Drive. Nếu dùng local runtime, đặt USE_GOOGLE_DRIVE=False.

Nguồn: https://github.com/googlecolab/colabtools/issues/6089


In [ ]:
import os
from pathlib import Path

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        from google.colab._message import MessageError
    except ImportError as exc:
        raise RuntimeError(
            "Kết nối bằng runtime Google Colab để dùng Drive. "
            "Nếu chạy local, đặt USE_GOOGLE_DRIVE=False."
        ) from exc

    mountpoint = Path("/content/drive")
    def drive_ready():
        return os.path.ismount(mountpoint) and (mountpoint / "MyDrive").is_dir()

    if not drive_ready():
        for attempt in range(2):
            try:
                print("Kết nối Google Drive: chọn tài khoản, chọn TẤT CẢ quyền được yêu cầu rồi Continue.")
                drive.mount(str(mountpoint), force_remount=(attempt == 1), timeout_ms=120000)
                if not drive_ready():
                    raise RuntimeError("Drive chưa có thư mục MyDrive sau khi mount.")
                break
            except MessageError as exc:
                raise RuntimeError(
                    "Colab chưa nhận được thông tin xác thực Google (MessageError). "
                    "Chạy lại cell này; trong hộp thoại đăng nhập, chọn đúng tài khoản "
                    "và đánh dấu tất cả quyền Drive/Colab được yêu cầu trước khi Continue. "
                    "Nếu không thấy hộp thoại, kiểm tra trình duyệt có chặn popup. "
                    "Nếu vẫn lỗi, dùng Runtime > Disconnect and delete runtime rồi kết nối lại. "
                    "Chưa tải dataset và chưa chuyển sang bộ nhớ tạm. "
                    f"Chi tiết gốc: {exc}"
                ) from exc
            except (ValueError, OSError, RuntimeError) as exc:
                if attempt == 0:
                    print("Kết nối chưa thành công; thử lại bằng force_remount=True.")
                else:
                    raise RuntimeError(
                        "Không kết nối được Google Drive sau 2 lần thử. "
                        "Nếu dùng local runtime, chuyển sang runtime Google Colab. "
                        "Trên Colab: chọn Runtime > Disconnect and delete runtime, "
                        "kết nối lại rồi Run All và cấp quyền Drive. "
                        "Để chạy tạm không dùng Drive, đặt USE_GOOGLE_DRIVE=False; "
                        "khi đó dữ liệu chỉ nằm trong phiên Colab. "
                        "Chưa tải dataset vì Drive chưa sẵn sàng."
                    ) from exc
    WORK_DIR = mountpoint / "MyDrive" / DRIVE_PROJECT_DIR
    print("Dữ liệu và kết quả sẽ được lưu trên Google Drive.")
else:
    WORK_DIR = Path("/content/parking-occupancy") if Path("/content").is_dir() else Path.cwd()
    print("Chế độ không dùng Drive. Trên Colab, dữ liệu sẽ mất khi runtime bị xóa.")

RAW_DIR = WORK_DIR / "data/raw"
DATASET_DIR = RAW_DIR / "PKLot"
OUTPUT_DIR = WORK_DIR / "outputs/full"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Thư mục lưu:", WORK_DIR)


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "Pillow>=10,<13"], check=True)


In [ ]:
import hashlib, importlib.util, json, sys, time, urllib.request
from urllib.parse import quote
assert OWNER != "YOUR_GITHUB_USERNAME", "Điền OWNER, REPO và REF ở cell cấu hình"
WORK_DIR.mkdir(parents=True, exist_ok=True)
url = (f"https://raw.githubusercontent.com/{quote(OWNER, safe='')}/"
       f"{quote(REPO, safe='')}/{quote(REF, safe='')}/{quote(MODULE_PATH, safe='/')}")
# Tải lại module mỗi lần Run All để lấy đúng REF đã cấu hình.
request = urllib.request.Request(
    f"{url}?refresh={time.time_ns()}", headers={"Cache-Control": "no-cache"}
)
with urllib.request.urlopen(request, timeout=60) as response:
    code = response.read()
compile(code, MODULE_PATH, "exec")
module_file = WORK_DIR / "parking_module.py"
module_file.write_bytes(code)
spec = importlib.util.spec_from_file_location("parking_module", module_file)
parking = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = parking
spec.loader.exec_module(parking)
provenance = {"url": url, "ref": REF, "sha256": hashlib.sha256(code).hexdigest()}
(WORK_DIR / "module_source.json").write_text(json.dumps(provenance, indent=2))
print("Loaded module:", provenance)


## Tải dữ liệu thật và kiểm tra

Mặc định archive, ảnh/XML đã giải nén và kết quả nằm trên Google Drive của bạn, trong MyDrive/parking-occupancy/. Dữ liệu được giữ qua các phiên Colab. Archive khoảng 4,9 GB; cần khoảng 10–15 GB trống. Lần sau tự dùng lại dữ liệu đã chuẩn bị thành công và chạy lại audit. Nguồn UFPR có thể tải chậm; giải nén/đọc nhiều file trên Drive cũng có thể mất thời gian.


In [ ]:
# provenance.json chỉ được ghi khi lần giải nén/audit trước đã hoàn tất.
prepared_before = (OUTPUT_DIR / "provenance.json").exists() and DATASET_DIR.is_dir()
if REUSE_EXTRACTED and prepared_before:
    print("Dùng lại dữ liệu đã giải nén; chạy lại audit.")
    summary = parking.audit(root=DATASET_DIR, output=OUTPUT_DIR)
else:
    summary = parking.prepare(raw=RAW_DIR, output=OUTPUT_DIR, download=DOWNLOAD_DATA)
print(json.dumps(summary, indent=2))
assert summary["valid_frames"] > 0


In [ ]:
# Đọc một mẫu hợp lệ trực tiếp bằng module lấy từ GitHub.
with (OUTPUT_DIR / "valid_samples.jsonl").open() as stream:
    sample = json.loads(next(stream))
frame = parking.read_frame(RAW_DIR / "PKLot" / sample["xml_path"], RAW_DIR / "PKLot")
print("Sample:", frame.sample_id)
print("Camera / weather / date:", frame.camera, frame.weather, frame.date)
print("Parking spaces:", len(frame.spaces))
print("Outputs:", OUTPUT_DIR)


## Nối bước tiếp theo

Gọi trực tiếp `parking.VehicleDetector`, `parking.OccupancyPrediction`, `parking.occupancy_metrics` khi triển khai detector và ghép ô. Polygon XML là vùng ô đỗ, không phải hộp xe. Chưa triển khai mô hình trong notebook này.
